# Week 6 – Prompt Engineering
## Designing Effective Prompts for Large Language Models

This notebook loads a pre-trained Hugging Face model and compares Zero-shot, One-shot, Few-shot, Role, and concise Chain-of-Thought prompting.

In [6]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

print("Project folder:", PROJECT_DIR)

Project folder: c:\Users\Lenovo T470s\Downloads\Prompt_Engineering_Week6\Prompt_Engineering


### 1. Install/import requirements
If you have already installed the requirements, the next cell can be skipped.

In [7]:
# In VS Code, run this once if needed:
# %pip install -r requirements.txt
print('If transformers is not installed, run: %pip install -r requirements.txt')

If transformers is not installed, run: %pip install -r requirements.txt


### 2. Load the project and model
The model is `google/flan-t5-small`, a pre-trained Hugging Face Transformer model.

In [8]:
from prompt_engineering import load_model, PROBLEMS, build_prompts, generate, score_response
model = load_model()
print('Model loaded successfully.')

Loading Hugging Face model: google/flan-t5-small


Device set to use cpu


Model loaded successfully.


### 3. Generate responses with five prompting techniques

In [9]:
import pandas as pd
rows = []
for problem_id, domain, problem in PROBLEMS:
    for technique, prompt in build_prompts(problem).items():
        response = generate(model, prompt)
        rows.append([problem_id, domain, technique, response])
comparison_df = pd.DataFrame(rows, columns=['problem_id','domain','technique','response'])
comparison_df.head(10)

,problem_id,domain,technique,response
0,1,Education,Zero-shot,Photosynthesis is a process that requires a lo...
1,1,Education,One-shot,Photosynthesis is a process that requires a lo...
2,1,Education,Few-shot,Answer: A student has difficulty understanding...
3,1,Education,Role,Photosynthesis is a process of photosynthesis.
4,1,Education,Chain-of-Thought,Identify the main issue.
5,2,Healthcare,Zero-shot,if you have a mild headache.
6,2,Healthcare,One-shot,Ask a doctor for advice on how to treat a head...
7,2,Healthcare,Few-shot,Answer: If a patient asks what they should do ...
8,2,Healthcare,Role,Ask a doctor about the symptoms of a headache.
9,2,Healthcare,Chain-of-Thought,Ask if they should go to a doctor's office.


### 4. Evaluate response quality
The rubric uses relevance, clarity, accuracy, and completeness. The scores are a simple automated aid; the responses should also be reviewed manually.

In [10]:
eval_rows = []
for _, row in comparison_df.iterrows():
    scores = score_response(row['response'], row['domain'])
    eval_rows.append([row['problem_id'], row['domain'], row['technique'], *scores])
evaluation_df = pd.DataFrame(eval_rows, columns=['problem_id','domain','technique','relevance','clarity','accuracy','completeness','total_score'])
evaluation_df.head(10)

,problem_id,domain,technique,relevance,clarity,accuracy,completeness,total_score
0,1,Education,Zero-shot,1,1,1,1,4
1,1,Education,One-shot,1,1,1,1,4
2,1,Education,Few-shot,2,1,2,1,6
3,1,Education,Role,1,1,1,1,4
4,1,Education,Chain-of-Thought,0,1,1,1,3
5,2,Healthcare,Zero-shot,1,1,1,1,4
6,2,Healthcare,One-shot,2,1,2,1,6
7,2,Healthcare,Few-shot,1,2,1,2,6
8,2,Healthcare,Role,2,1,2,1,6
9,2,Healthcare,Chain-of-Thought,1,1,1,1,4


### 5. Save CSV outputs

In [11]:
from pathlib import Path
dataset = Path('dataset')
dataset.mkdir(exist_ok=True)
comparison_df.to_csv(dataset/'prompt_comparison.csv', index=False)
comparison_df[['problem_id','technique','response']].to_csv(dataset/'generated_responses.csv', index=False)
evaluation_df.to_csv(dataset/'evaluation_results.csv', index=False)
print('CSV files created successfully.')

CSV files created successfully.


### 6. Conclusion
- Zero-shot gives instructions without examples.
- One-shot provides one example to guide style and format.
- Few-shot provides multiple examples and can improve consistency.
- Role prompting sets an appropriate perspective.
- Concise CoT-style structure asks for an organized reasoning summary without exposing hidden reasoning.

Prompt engineering matters because clearer instructions and examples can make model outputs more relevant, consistent, and useful.